[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-07-i-o-parquet-csv-json-databases.ipynb#scrollTo=c1e2f3a4)

---
# Day 7 · I/O: Parquet, CSV, JSON, Databases
**certified-journeys / polars-certified** &nbsp;|&nbsp; Intermediate

> **Goal for today:** Read and write all major file formats efficiently, use predicate pushdown with Parquet, and connect Polars to DuckDB.

---
## Why I/O strategy matters

The file format you choose determines query speed and storage cost far more than any other decision.

| Format | Read speed | Write speed | File size | Predicate pushdown | Best for |
|---|---|---|---|---|---|
| CSV | Slow | Fast | Large | No | Interchange, small files |
| JSON/NDJSON | Slow | Slow | Largest | No | APIs, nested data |
| Parquet | Fast | Fast | Small | **Yes** | Analytics, production |
| Parquet (partitioned) | Fastest | Medium | Smallest per query | **Yes + partition pruning** | TB-scale analytics |

In [ ]:
%pip install -q polars pyarrow duckdb

---
## Step 1 · Generate a 200k-row dataset, compare CSV vs Parquet sizes

We'll generate a realistic synthetic dataset and measure how much space each format uses.
The difference is usually 5–10x — crucial at scale.

In [ ]:
import polars as pl
import numpy as np
import os
import tempfile

rng = np.random.default_rng(42)
N = 200_000

# Simulate a sales events table
df = pl.DataFrame({
    "event_id":   np.arange(N),
    "customer_id": rng.integers(1, 5000, N),
    "product_id":  rng.integers(1, 500, N),
    "amount":      rng.uniform(5.0, 500.0, N).round(2),
    "year":        rng.choice([2022, 2023, 2024], N).tolist(),
    "month":       rng.integers(1, 13, N).tolist(),
    "region":      rng.choice(["US", "EU", "APAC", "LATAM"], N).tolist(),
    "status":      rng.choice(["completed", "pending", "refunded"], N).tolist(),
})

# Use a temp directory so Colab stays clean
TMPDIR = tempfile.mkdtemp()
csv_path = os.path.join(TMPDIR, "sales.csv")
parquet_path = os.path.join(TMPDIR, "sales.parquet")

df.write_csv(csv_path)
df.write_parquet(parquet_path)  # default compression: zstd

csv_mb    = os.path.getsize(csv_path)    / 1_048_576
parquet_mb = os.path.getsize(parquet_path) / 1_048_576
print(f"CSV:     {csv_mb:.2f} MB")
print(f"Parquet: {parquet_mb:.2f} MB")
print(f"Parquet is {csv_mb/parquet_mb:.1f}x smaller than CSV")

**What just happened?**
- Parquet is typically **5–10x smaller** than CSV for the same data because it uses columnar storage + compression.
- Polars defaults to `zstd` compression for Parquet — a great balance of speed and ratio.
- The `region` and `status` columns compress especially well because they have low cardinality (repeated strings).
- **This size difference compounds**: at 1 TB of CSV, Parquet saves ~900 GB of storage and network transfer costs.

---
## Step 2 · read_csv vs scan_csv — eager vs lazy I/O

The most important I/O decision: **load into memory now, or defer until needed?**

| | `read_csv()` | `scan_csv()` |
|---|---|---|
| Returns | `DataFrame` (in RAM) | `LazyFrame` (query plan) |
| Memory usage | Full file size | Near-zero until `.collect()` |
| Can push down filters | No | Yes |
| Can sample with `n_rows` | Yes | Yes |

In [ ]:
# Eager: loads entire file into a DataFrame immediately
df_eager = pl.read_csv(csv_path)
print(f"read_csv result type:  {type(df_eager)}")
print(f"Rows loaded eagerly:   {df_eager.shape[0]:,}")

# Lazy: returns a query plan, no data loaded yet
lf_lazy = pl.scan_csv(csv_path)
print(f"\nscan_csv result type:  {type(lf_lazy)}")

# Sample the first 100 rows without reading the whole file
sample = pl.scan_csv(csv_path, n_rows=100).collect()
print(f"scan_csv sample shape: {sample.shape}")

# Lazy filter: only the matching rows are fully read
us_revenue = (
    pl.scan_csv(csv_path)
    .filter(pl.col("region") == "US")
    .select(["customer_id", "amount", "region"])
    .collect()
)
print(f"\nUS rows via scan_csv:  {us_revenue.shape[0]:,}")

**What just happened?**
- `scan_csv()` returns a `LazyFrame` instantly — no disk I/O until `.collect()` is called.
- **Filter pushdown on CSV**: Polars still reads the full CSV (CSV has no metadata for skipping rows), but avoids materialising filtered-out rows into memory.
- `n_rows=100` is perfect for quickly sampling a large file to inspect its schema before committing to a full load.
- For true predicate pushdown (skipping actual disk reads), use **Parquet** — demonstrated in Step 4.

---
## Step 3 · Parquet compression options

Polars supports multiple compression codecs for Parquet.
The right choice depends on whether you optimise for file size, write speed, or read speed.

In [ ]:
import time

compressions = ["snappy", "zstd", "lz4"]

for codec in compressions:
    path = os.path.join(TMPDIR, f"sales_{codec}.parquet")
    t0 = time.perf_counter()
    df.write_parquet(path, compression=codec)
    write_ms = (time.perf_counter() - t0) * 1000
    size_mb = os.path.getsize(path) / 1_048_576
    print(f"{codec:8s}: {size_mb:.2f} MB  ({write_ms:.0f} ms write)")

# Summary of tradeoffs:
print("""
Compression tradeoffs:
  snappy  → fast read/write, moderate compression   (good default for hot data)
  zstd    → smaller files, slightly slower write     (Polars default, best for cold storage)
  lz4     → fastest read/write, least compression   (best for real-time pipelines)
""")

**What just happened?**
- All three codecs produce files much smaller than CSV — Parquet's columnar layout does most of the compression work before the codec is even applied.
- **`zstd`** is Polars' default and the right choice for archival or cold storage workloads.
- **`lz4`** is fastest end-to-end and recommended when write throughput is the bottleneck (streaming ingestion).
- **`snappy`** is the Spark/Hadoop ecosystem default — use it when sharing files across tools.

---
## Step 4 · scan_parquet with predicate pushdown

Unlike CSV, Parquet stores **row-group statistics** (min/max per column per chunk).
Polars uses these to skip entire row groups that can't contain matching rows — this is predicate pushdown.

In [ ]:
# Lazy Parquet scan with a filter applied BEFORE reading data
lf_parquet = (
    pl.scan_parquet(parquet_path)
    .filter(pl.col("year") == 2024)
    .select(["event_id", "customer_id", "amount", "year", "region"])
)

# explain() shows the query plan — look for SELECTION and PROJECT lines
print("=== Query plan (optimized) ===")
print(lf_parquet.explain(optimized=True))

# Collect: Polars reads only the necessary row groups from disk
result = lf_parquet.collect()
print(f"\nRows for 2024: {result.shape[0]:,} (out of {N:,} total)")

**What just happened?**
- The query plan shows `SELECTION` (predicate pushdown) and `PROJECT` (column pruning) applied at the Parquet reader level.
- Polars reads **only the columns you asked for** and **only the row groups that might contain `year == 2024`**.
- This is why Parquet + `scan_parquet` is the gold standard: the filter reduces I/O, not just memory usage.
- **For best pushdown performance**, sort your Parquet file by the filter column before writing — maximises row group skipping.

---
## Step 5 · Partitioned Parquet (Hive-style)

Partitioning splits a dataset into separate files by a column value (e.g., one file per year).
Polars can then skip entire files when their partition column doesn't match the filter.

In [ ]:
import pathlib

partition_dir = os.path.join(TMPDIR, "sales_partitioned")
pathlib.Path(partition_dir).mkdir(exist_ok=True)

# Write one Parquet file per year (Hive-style partitioning)
for year, partition_df in df.group_by("year"):
    year_val = year[0]  # group_by returns tuples
    year_dir = os.path.join(partition_dir, f"year={year_val}")
    pathlib.Path(year_dir).mkdir(exist_ok=True)
    partition_df.write_parquet(os.path.join(year_dir, "data.parquet"))
    print(f"Written year={year_val}: {partition_df.shape[0]:,} rows")

# Read all partitions with a glob pattern
all_partitions = pl.scan_parquet(os.path.join(partition_dir, "**/*.parquet")).collect()
print(f"\nAll partitions loaded: {all_partitions.shape[0]:,} rows")

# Scan only the 2024 partition by pointing at the specific folder
only_2024 = pl.scan_parquet(
    os.path.join(partition_dir, "year=2024/*.parquet")
).collect()
print(f"Only 2024 partition:   {only_2024.shape[0]:,} rows")

**What just happened?**
- We simulated **Hive-style partitioning**: `year=2022/data.parquet`, `year=2023/data.parquet`, `year=2024/data.parquet`.
- The `**/*.parquet` glob pattern reads all partitions at once — Polars handles the file traversal automatically.
- Pointing at a **specific partition folder** (`year=2024/*.parquet`) avoids reading other years entirely — this is **partition pruning** on top of predicate pushdown.
- At TB scale, partition pruning is often the single biggest performance lever available.

---
## Step 6 · JSON and NDJSON with nested data

Polars can read JSON arrays and newline-delimited JSON (NDJSON).
Nested objects are automatically parsed into `Struct` columns.

In [ ]:
import json

# Write NDJSON with nested data (one JSON object per line)
ndjson_path = os.path.join(TMPDIR, "events.ndjson")
events = [
    {"id": 1, "type": "click", "user": {"id": 101, "country": "US"}, "value": 1.5},
    {"id": 2, "type": "view",  "user": {"id": 102, "country": "UK"}, "value": 0.0},
    {"id": 3, "type": "buy",   "user": {"id": 101, "country": "US"}, "value": 49.99},
]
with open(ndjson_path, "w") as f:
    for e in events:
        f.write(json.dumps(e) + "\n")

# Read NDJSON — nested 'user' object becomes a Struct column
df_json = pl.read_ndjson(ndjson_path)
print("NDJSON loaded:")
print(df_json)
print("\nSchema:", df_json.schema)

# Flatten the Struct by using .struct.field() or .unnest()
df_flat = df_json.with_columns([
    pl.col("user").struct.field("id").alias("user_id"),
    pl.col("user").struct.field("country").alias("user_country"),
]).drop("user")
print("\nFlattened:")
print(df_flat)

**What just happened?**
- Polars parsed the nested `user` object as a **`Struct` dtype** automatically — no manual parsing needed.
- `.struct.field("key")` extracts a sub-field from a Struct column as its own Series.
- `.unnest("user")` is the shorthand to expand **all** struct fields into separate columns at once.
- **NDJSON is preferred over JSON arrays** for large files: each line is independent, enabling streaming reads.

---
## Step 7 · DuckDB integration

DuckDB can query Polars DataFrames directly via SQL — zero copy, no serialisation.
This gives you the full SQL surface area (window functions, CTEs, etc.) on Polars data.

In [ ]:
import duckdb

# Create an in-memory DuckDB connection
con = duckdb.connect()

# Register a Polars DataFrame as a virtual DuckDB table (zero copy)
con.register("sales", df)

# Run a SQL query — DuckDB operates directly on the Polars memory
sql_result = con.execute("""
    SELECT
        region,
        year,
        COUNT(*)          AS num_orders,
        ROUND(SUM(amount), 2)  AS total_revenue,
        ROUND(AVG(amount), 2)  AS avg_order_value
    FROM sales
    WHERE status = 'completed'
    GROUP BY region, year
    ORDER BY region, year
""").pl()  # .pl() returns the result as a Polars DataFrame

print("DuckDB SQL result (as Polars DataFrame):")
print(sql_result)
print(f"\nResult type: {type(sql_result)}")

**What just happened?**
- `con.register("sales", df)` exposes the Polars DataFrame to DuckDB **without copying data** — both share the same Arrow memory.
- `.pl()` on a DuckDB result returns a Polars DataFrame directly — no intermediate Pandas step needed.
- **DuckDB is ideal when** you have complex SQL (window functions, recursive CTEs, lateral joins) that would be verbose to express in Polars expressions.
- The Polars → DuckDB → Polars round-trip is nearly zero-cost and is a common production pattern.

---
## Challenge

1. Load `sales.csv` lazily with `scan_csv()`
2. Filter for `status == "completed"` and `year >= 2023` **without collecting**
3. Write the filtered result to partitioned Parquet, one file per `year`
4. Scan only the `year=2024` partition with an additional filter on `region == "US"`
5. Print the shape of the final result

In [ ]:
challenge_dir = os.path.join(TMPDIR, "challenge_partitions")
pathlib.Path(challenge_dir).mkdir(exist_ok=True)

# TODO: Step 1+2 — lazy scan and filter
filtered_lf = ...  # scan_csv + filter (no .collect() yet)

# TODO: Step 3 — collect and write partitioned by year
# Hint: filtered_lf.collect(), then group_by("year"), then write_parquet per partition

# TODO: Step 4 — scan only the 2024 partition with a region filter
result = ...  # scan_parquet(...).filter(...).collect()

# print(result.shape)

---
## Day 7 recap

| Concept | Key method | When to use |
|---|---|---|
| Eager CSV read | `pl.read_csv(path)` | Small files, exploration |
| Lazy CSV scan | `pl.scan_csv(path)` | Large files, filter before load |
| Write Parquet | `df.write_parquet(path, compression=)` | Always for production storage |
| Lazy Parquet scan | `pl.scan_parquet(path)` | Predicate + projection pushdown |
| Partitioned Parquet | Write per group, glob to read | TB-scale datasets |
| NDJSON | `pl.read_ndjson(path)` | API responses, nested data |
| DuckDB integration | `con.register("t", df)` + `.pl()` | Complex SQL on Polars data |

> **Tip:** Parquet + Polars is the fastest stack for analytical workloads. Use partitioned Parquet for TB-scale data — partition pruning avoids reading files entirely, not just filtering rows within them.

---
**What's next → Day 8:** Performance optimisation and profiling — `.explain()`, streaming mode, rechunking, and benchmarking.

Mark Day 7 complete in your [tracker](../index.html).